# Tools y Skills

Input 3 → Tool increment resuelta en la Skill → output 4.

API **2.1.3**. Cada ejemplo muestra una entrada y su resultado. Python Runtime realiza trabajo determinista.

In [ ]:
import agentic_systems as toolkit


In [ ]:
@toolkit.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@toolkit.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = toolkit.skill(name="arithmetic", tools=[increment, double])
increase = toolkit.Action("increase", "increment")
multiply = toolkit.Action("multiply", "double")

In [ ]:
agent = toolkit.Agent(name="incrementer", flow=increase, skills=(arithmetic,))
result = agent.run({"value": 3})
assert result.ok and result.data == {"value": 4}
print({"input": {"value": 3}, "process": "Skill arithmetic → increment", "output": result.data})

Una Skill no se ejecuta. Se asocia con `skills=(arithmetic,)`; Action resuelve la Tool por nombre. `load_skill(path)` carga una Skill portable escrita con esta misma API; requiere `SKILL.md` y `skill.py` con `build_skill()`. Es carga desde archivo, no una segunda gramática de construcción.

Los parámetros canónicos de contrato son input_schema y output_schema. Este ejemplo usa modelos Pydantic con StrictInt. Tool expone run, arun e inspect; no tiene compile. Una Skill expone instrucciones y capacidades, no otro ejecutor.

In [ ]:
from lesson_examples import validated_increment as validated
assert validated.run({'value':3}).data == {'value':4}
assert (await validated.arun({'value':3})).data == {'value':4}
assert not validated.run({'value':'invalid'}).ok
print(validated.inspect())

In [ ]:
from portable_skills.example import run as portable_example
report = portable_example()
print({'input':report['input'], 'process':report['process'], 'output':report['output']['data']})

In [ ]:
from pathlib import Path
import tempfile
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder)
    (path / "SKILL.md").write_text("# Arithmetic", encoding="utf-8")
    (path / "skill.py").write_text("import agentic_systems as toolkit\n@toolkit.tool\ndef inc(value:int)->dict:\n    return {'value':value+1}\ndef build_skill():\n    return toolkit.skill(name='loaded',tools=[inc])\n", encoding="utf-8")
    loaded = toolkit.load_skill(path)
    loaded_agent = toolkit.Agent("loaded", toolkit.Action("inc", "inc"), skills=(loaded,))
    assert loaded_agent.run({"value": 3}).data == {"value": 4}
print("Skill portable cargada y ejecutada mediante Agent")